# 05 — Transações, Integridade e PostgreSQL

**Objetivos:** entender atomicidade e isolamento e experimentar servidor real quando disponível

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia Técnico](./reference_materials/05_transacoes_postgresql.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_transacoes_postgresql.pdf)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## ACID na prática

**Atomicidade:** tudo ou nada. **Consistência:** preservação de invariantes do banco. **Isolamento:** transações concorrentes devem seguir nível especificado. **Durabilidade:** alterações confirmadas persistem após falhas; efeitos dependem de configurações do SGBD.

In [2]:
import sqlite3
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys=ON')
conn.executescript((Path('sql')/'01_schema_sqlite.sql').read_text(encoding='utf8'))
leituras = pd.read_csv(DATA/'leituras_sensores.csv')
leituras.to_sql('leituras',conn,if_exists='append',index=False)
print('SQLite inicializado:',conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0], 'leituras')


SQLite inicializado: 72 leituras


## Rollback local no SQLite

Demonstração de cancelamento de inserção; SQLite não substitui os laboratórios de concorrência do PostgreSQL.

In [3]:
antes=conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0]
conn.execute('BEGIN') if not conn.in_transaction else conn.commit()
conn.execute("INSERT INTO leituras VALUES (999,1,'2025-02-01',22,50)")
conn.rollback()
depois=conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0]
print('Linhas antes/depois:',antes,depois);assert antes==depois

Linhas antes/depois: 72 72


## Prática opcional — PostgreSQL real

Ative o serviço com `docker compose -f infra/compose.yaml --profile postgres up -d` e configure `POSTGRES_PASSWORD` via `.env` na pasta infra. O código abaixo **não declara sucesso** sem conexão; faz uma checagem explícita e só executa quando `RUN_LIVE_DB=1`.

In [4]:
# A execução local não ativa conexões automaticamente.
# Para laboratórios opcionais, configure RUN_LIVE_DB=1 antes de iniciar o kernel,
# depois de verificar que a instância é EXCLUSIVAMENTE de estudos.
import os
print('Laboratório externo:', 'habilitado' if os.getenv('RUN_LIVE_DB') == '1' else 'desabilitado (padrão seguro)')

Laboratório externo: desabilitado (padrão seguro)


In [5]:
import os
if os.getenv('RUN_LIVE_DB')=='1':
 import psycopg
 senha=os.getenv('POSTGRES_PASSWORD')
 assert senha,'Defina POSTGRES_PASSWORD no ambiente do Jupyter.'
 with psycopg.connect(host='localhost',port=5433,user=os.getenv('POSTGRES_USER','estudos'),password=senha,dbname=os.getenv('POSTGRES_DB','biblioteca'),connect_timeout=5) as pg:
  print('Versão real:',pg.execute('SELECT version()').fetchone()[0])
else:
 print('Prática real não executada. Defina RUN_LIVE_DB=1 com o serviço ativo.')

Prática real não executada. Defina RUN_LIVE_DB=1 com o serviço ativo.


## Exercício resolvido — transferência

Faça uma transferência entre contas somente se o saldo de origem for suficiente; use transação e CHECK(saldo >= 0). O script `sql/03_postgres.sql` é uma base de estudo, não representa tratamento completo de concorrência.

In [6]:
print((Path('sql')/'03_postgres.sql').read_text()[:400])

-- Executar apenas no PostgreSQL; sessão demonstrativa de bloqueio
BEGIN;
CREATE TABLE IF NOT EXISTS contas_demo (id INTEGER PRIMARY KEY, saldo NUMERIC(10,2) NOT NULL CHECK(saldo>=0));
INSERT INTO contas_demo(id,saldo) VALUES(1,100),(2,60) ON CONFLICT(id) DO NOTHING;
SAVEPOINT antes_transferencia;
UPDATE contas_demo SET saldo=saldo-20 WHERE id=1;
UPDATE contas_demo SET saldo=saldo+20 WHERE id=2;
C


## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Atomicidade e idempotência

Uma transação garante atomicidade das operações no mesmo banco, mas não substitui modelagem de idempotência nem controle de concorrência. Em PostgreSQL, verifique o nível de isolamento e use tabelas exclusivamente de laboratório. O bloco abaixo funciona sem servidor e não altera arquivos persistentes.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [7]:
import sqlite3
with sqlite3.connect(':memory:',isolation_level=None) as db:
    db.execute('CREATE TABLE contas(id INTEGER PRIMARY KEY, saldo INTEGER CHECK(saldo>=0))')
    db.executemany('INSERT INTO contas VALUES (?,?)',[(1,100),(2,30)])
    db.execute('BEGIN')
    try:
        db.execute('UPDATE contas SET saldo=saldo-200 WHERE id=1')
        db.execute('UPDATE contas SET saldo=saldo+200 WHERE id=2')
        db.execute('COMMIT')
    except sqlite3.IntegrityError:
        db.execute('ROLLBACK')
    assert db.execute('SELECT SUM(saldo) FROM contas').fetchone()[0] ==130
    print('Rollback preservou os saldos:',db.execute('SELECT * FROM contas ORDER BY id').fetchall())

Rollback preservou os saldos: [(1, 100), (2, 30)]


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [8]:
pd.DataFrame({'metrica':['registros_apos_rollback'], 'valor':[conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0]]}).to_csv(OUT / '05_validacao_rollback.csv',index=False)
print('Exportado: outputs/05_validacao_rollback.csv')

Exportado: outputs/05_validacao_rollback.csv
